# Colab Training Notebook

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
!pip install -q torchmetrics mlflow tqdm

In [11]:
import torch
import shutil, os, glob
import mlflow
import pandas as pd
import sys
from dotenv import load_dotenv
from pathlib import Path

In [12]:
load_dotenv()

True

In [13]:
# Check GPU
if torch.cuda.is_available():
    print(f"CUDA available: {torch.cuda.is_available()}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU")

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [14]:
# Set Drive path
DRIVE_PATH = Path(os.environ["DRIVE_PATH"])
os.makedirs(f'{DRIVE_PATH}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/data/splits', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/data/labeled', exist_ok=True)
print("Drive path ready")

Drive path ready


In [15]:
if str(DRIVE_PATH) not in sys.path:
    sys.path.append(str(DRIVE_PATH))

from src.train import train_model
from src.evaluate import evaluate 
from src.alignment_head import train_alignment_head

from src.image_model import AdImageModel
from src.text_encoder import TextEncoder
from src.alignment import compute_alignment
from src.caption_generator import CaptionGenerator
from src.colors import extract_dominant_colors
from src.platform_guides import PLATFORM_GUIDES


In [16]:
# Set MLflow tracking URI to local SQLite database
mlflow.set_tracking_uri(f"sqlite:///{DRIVE_PATH}/mlflow.db")
mlflow.set_experiment("visiomark-image-model")
print("MLflow ready")

MLflow ready


In [17]:
# Check data splits
splits = {}
for split in ['train', 'val', 'test']:
    path = f"{DRIVE_PATH}/data/splits/{split}.csv"
    if os.path.exists(path):
        df = pd.read_csv(path)
        splits[split] = df
        print(f"{split}: {len(df)} samples")
        print(f"content_type distribution: {df['content_type_label'].value_counts().to_dict()}")
        print(f"mood distribution:         {df['mood_label'].value_counts().to_dict()}")
        print()
    else:
        print(f"{split}.csv not found")

train: 1131 samples
content_type distribution: {1: 609, 2: 355, 0: 167}
mood distribution:         {1: 423, 2: 352, 0: 224, 3: 132}

val: 243 samples
content_type distribution: {1: 131, 2: 76, 0: 36}
mood distribution:         {1: 91, 2: 76, 0: 47, 3: 29}

test: 243 samples
content_type distribution: {1: 131, 2: 76, 0: 36}
mood distribution:         {1: 90, 2: 76, 0: 48, 3: 29}



### Phase 1

In [18]:
print("Starting Phase 1.")
print("=" * 60)

checkpoints_path = train_model(
    phase=1,
    epochs=15,
    batch_size=16,
    lr=1e-3,
    img_dir=f"{DRIVE_PATH}/data/raw/",
    train_csv=f"{DRIVE_PATH}/data/splits/train.csv",
    val_csv=f"{DRIVE_PATH}/data/splits/val.csv",
    checkpoint=None,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    save_dir=f"{DRIVE_PATH}/checkpoints",
)

print("Phase 1 checkpoints saved to: /checkpoints ")

Starting Phase 1.
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 192MB/s]


Epoch 01/15 | loss=2.3382 | F1_content=0.6493 | F1_mood=0.4829 | Avg_F1=0.5661


2026/06/09 22:11:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/09 22:11:31 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/09 22:11:31 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/09 22:11:43 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version

    Checkpoint saved (avg_F1=0.5661)


...

    Checkpoint saved (avg_F1=0.6222)
Epoch 09/15 | loss=1.6145 | F1_content=0.6600 | F1_mood=0.5701 | Avg_F1=0.6151
Epoch 10/15 | loss=1.7007 | F1_content=0.6291 | F1_mood=0.5568 | Avg_F1=0.5929
Epoch 11/15 | loss=1.6182 | F1_content=0.6574 | F1_mood=0.5647 | Avg_F1=0.6110
Epoch 12/15 | loss=1.6452 | F1_content=0.6742 | F1_mood=0.5655 | Avg_F1=0.6199


2026/06/09 22:17:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch 13/15 | loss=1.6125 | F1_content=0.6654 | F1_mood=0.5846 | Avg_F1=0.6250


2026/06/09 22:17:03 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/09 22:17:03 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/09 22:17:11 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchvision==0.26.0' without the loca

    Checkpoint saved (avg_F1=0.6250)
Epoch 14/15 | loss=1.6163 | F1_content=0.6709 | F1_mood=0.5724 | Avg_F1=0.6216
Epoch 15/15 | loss=1.5729 | F1_content=0.6666 | F1_mood=0.5718 | Avg_F1=0.6192

Phase 1 done. Best avg F1 = 0.6250
Phase 1 checkpoints saved to: /checkpoints 


### Phase 2

In [19]:
print("Starting Phase 2.")
print("=" * 60)

phase1_checkpoints = sorted(glob.glob(f'{DRIVE_PATH}/checkpoints/best_phase1*.pt'))

if not phase1_checkpoints:
    raise FileNotFoundError("No Phase 1 checkpoint found.")

best_phase1 = phase1_checkpoints[-1]

final_checkpoints_path = train_model(
    phase=2,
    epochs=50,
    batch_size=16,
    lr=1e-5,
    img_dir=f"{DRIVE_PATH}/data/raw/",
    train_csv=f"{DRIVE_PATH}/data/splits/train.csv",
    val_csv=f"{DRIVE_PATH}/data/splits/val.csv",
    checkpoint=best_phase1,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    save_dir=f"{DRIVE_PATH}/checkpoints",
)

print("Phase 2 checkpoints saved to: /checkpoints ")

Starting Phase 2.
  Loading Phase 1 checkpoint
  Unfroze last 3 encoder blocks
Epoch 01/50 | loss=1.6167 | F1_content=0.6674 | F1_mood=0.5789 | Avg_F1=0.6231


2026/06/09 22:18:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/09 22:18:29 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/09 22:18:29 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/09 22:18:37 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version

    Checkpoint saved (avg_F1=0.6231)


2026/06/09 22:19:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch 02/50 | loss=1.6083 | F1_content=0.6711 | F1_mood=0.5772 | Avg_F1=0.6242


...

    Checkpoint saved (avg_F1=0.6877)
Epoch 35/50 | loss=1.0680 | F1_content=0.7187 | F1_mood=0.6446 | Avg_F1=0.6816


2026/06/09 22:35:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch 36/50 | loss=1.0412 | F1_content=0.7278 | F1_mood=0.6580 | Avg_F1=0.6929


2026/06/09 22:35:13 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/09 22:35:14 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/09 22:35:21 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchvision==0.26.0' without the loca

    Checkpoint saved (avg_F1=0.6929)
Epoch 37/50 | loss=1.0071 | F1_content=0.7203 | F1_mood=0.6431 | Avg_F1=0.6817
Epoch 38/50 | loss=1.0495 | F1_content=0.7185 | F1_mood=0.6372 | Avg_F1=0.6778
Epoch 39/50 | loss=0.9930 | F1_content=0.7134 | F1_mood=0.6310 | Avg_F1=0.6722
Epoch 40/50 | loss=1.0229 | F1_content=0.7198 | F1_mood=0.6590 | Avg_F1=0.6894
Epoch 41/50 | loss=1.0423 | F1_content=0.7202 | F1_mood=0.6454 | Avg_F1=0.6828
Epoch 42/50 | loss=1.0445 | F1_content=0.7059 | F1_mood=0.6690 | Avg_F1=0.6874
Epoch 43/50 | loss=1.0184 | F1_content=0.7224 | F1_mood=0.6146 | Avg_F1=0.6685
Epoch 44/50 | loss=1.0293 | F1_content=0.7167 | F1_mood=0.6454 | Avg_F1=0.6810
Epoch 45/50 | loss=1.0329 | F1_content=0.7147 | F1_mood=0.6335 | Avg_F1=0.6741
Epoch 46/50 | loss=1.0460 | F1_content=0.7103 | F1_mood=0.6282 | Avg_F1=0.6692
Epoch 47/50 | loss=1.0112 | F1_content=0.7144 | F1_mood=0.6653 | Avg_F1=0.6898
Epoch 48/50 | loss=1.0071 | F1_content=0.7175 | F1_mood=0.6636 | Avg_F1=0.6905
Epoch 49/50 | l

## Evaluation

In [20]:
phase2_checkpoints = sorted(glob.glob(f'{DRIVE_PATH}/checkpoints/best_phase2*.pt'))

In [21]:
checkpoint_path = phase2_checkpoints[-1] if phase2_checkpoints else None
if checkpoint_path is None:
    raise FileNotFoundError("No Phase 2 checkpoint found for evaluation.")
print("Checkpoint ready for evaluation")

Checkpoint ready for evaluation


In [22]:
result = evaluate(
    checkpoint=checkpoint_path,
    img_dir=f"{DRIVE_PATH}/data/raw/",
    test_csv=f"{DRIVE_PATH}/data/splits/test.csv",
    save_path=f"{DRIVE_PATH}/eval/final_results.json",
)
print("Results saved to eval/final_results.json")

Loading model on cuda
Loading test data

TEST RESULTS
Content Type F1 : 0.7048
Mood F1         : 0.5984
Average F1      : 0.6516

Confusion Matrices

Content Type Confusion Matrix:

Mood Confusion Matrix:
Saved 'eval/content_type_confusion_matrix.png' and 'eval/mood_confusion_matrix.png'

Content Type Classification Report:
                  precision    recall  f1-score   support

Product Showcase       0.43      0.56      0.49        36
       Lifestyle       0.76      0.73      0.75       131
     Promotional       0.76      0.71      0.73        76

        accuracy                           0.70       243
       macro avg       0.65      0.67      0.66       243
    weighted avg       0.71      0.70      0.70       243


Mood Classification Report:
              precision    recall  f1-score   support

   Energetic       0.47      0.48      0.47        48
        Calm       0.68      0.60      0.64        90
Professional       0.71      0.64      0.68        76
     Playful       

### Extract Image Vectors for Alignment Training


In [23]:
!pip install -q torchmetrics transformers tqdm

#### Image Vectors

In [24]:
# Extract image vectors from alignment pairs
import os
import torch
import pandas as pd
from PIL import Image
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load model
model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1).to(DEVICE)
model.classifier = torch.nn.Identity() 
model.eval()

# Load alignment pairs CSV
# Expected columns: image_name, caption
ALIGNMENT_CSV = f'{DRIVE_PATH}/data/alignment_pairs/alignment_pairs.csv'

if not os.path.exists(ALIGNMENT_CSV):
    print("alignment_pairs.csv not found.")
    print("Upload data/alignment_pairs/ folder first.")
else:
    df = pd.read_csv(ALIGNMENT_CSV)
    print(f"Loaded {len(df)} alignment pairs")

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    image_vectors = []
    valid_idx = []
    cleaned_captions = []
    
    print("Extracting vectors for manual pairs")

    with torch.no_grad():
        for idx, row in tqdm(df.iterrows(), total=len(df), desc="Extracting"):
            caption = str(row.get("caption", "")).strip()
            words = caption.split()
            word_count = len(words)
            
            if word_count < 8:
                continue  
            
            if word_count > 25:
                words = words[:25]
                caption = " ".join(words)
            
            img_path = f'{DRIVE_PATH}/data/raw/{row["image_name"]}'
            try:
                img = Image.open(img_path).convert('RGB')
                img_t = transform(img).unsqueeze(0).to(DEVICE)
                vec = model(img_t).squeeze(0).cpu()
                
                image_vectors.append(vec)
                valid_idx.append(idx)
                cleaned_captions.append(caption)
            except Exception as e:
                print(f"  Skip {row['image_name']}: {e}")


    image_vectors = torch.stack(image_vectors)  # [N, 1280]
    df_valid = df.iloc[valid_idx].reset_index(drop=True)
    
    df_valid["caption"] = cleaned_captions

    # Save
    torch.save(image_vectors, f'{DRIVE_PATH}/checkpoints/alignment_image_vectors.pt')
    df_valid.to_csv(f'{DRIVE_PATH}/data/alignment_pairs/alignment_pairs_valid.csv', index=False)

    print(f"\n {len(image_vectors)} image vectors saved → Drive")
    print(f"   Shape: {image_vectors.shape}")
    print(f"   Filtered out: {len(df) - len(df_valid)} rows")

Loaded 201 alignment pairs
Extracting vectors for manual pairs


Extracting: 100%|██████████| 201/201 [02:40<00:00,  1.25it/s]



 193 image vectors saved → Drive
   Shape: torch.Size([193, 1280])
   Filtered out: 8 rows


##### Text Vectors

In [25]:
tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
text_model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2').to(DEVICE)
text_model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 384, padding_idx=0)
    (position_embeddings): Embedding(512, 384)
    (token_type_embeddings): Embedding(2, 384)
    (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-5): 6 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
    

In [26]:
text_vectors = []

with torch.no_grad():
    for caption in tqdm(df_valid["caption"].tolist(), desc="Encoding captions"):
        inputs = tokenizer(caption, return_tensors="pt", truncation=True, max_length=128, padding=True).to(DEVICE)
        out = text_model(**inputs)
        attention_mask = inputs["attention_mask"]
        token_embeddings = out.last_hidden_state
        mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        emb = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        emb = torch.nn.functional.normalize(emb, dim=-1).squeeze(0).cpu()
        text_vectors.append(emb)

text_vectors = torch.stack(text_vectors)  # [N, 384]
torch.save(text_vectors, f'{DRIVE_PATH}/checkpoints/alignment_text_vectors.pt')
print(f"Text vectors saved → shape: {text_vectors.shape}")

Encoding captions: 100%|██████████| 193/193 [00:01<00:00, 158.35it/s]


Text vectors saved → shape: torch.Size([193, 384])


In [27]:
img_vectors = torch.load(f'{DRIVE_PATH}/checkpoints/alignment_image_vectors.pt', map_location='cpu',weights_only=True)
txt_vectors = torch.load(f'{DRIVE_PATH}/checkpoints/alignment_text_vectors.pt', map_location='cpu',weights_only=True)

In [28]:
alignment_head = train_alignment_head(
    image_vectors=img_vectors,
    text_vectors=txt_vectors,
    epochs=40,
    batch_size=16,
    lr=1e-3,
    device='cuda',
    save_path=f'{DRIVE_PATH}/checkpoints/alignment_head.pt',
)

Epoch 01 | train_loss=2.8038 | val_loss=2.3801
 Best checkpoint (val_loss=2.3801)
Epoch 02 | train_loss=0.6232 | val_loss=2.5306
Epoch 03 | train_loss=0.1610 | val_loss=3.0496
Epoch 04 | train_loss=0.0308 | val_loss=3.4055
Epoch 05 | train_loss=0.0072 | val_loss=3.9073
Epoch 06 | train_loss=0.0020 | val_loss=4.2576
Epoch 07 | train_loss=0.0071 | val_loss=4.3990
Epoch 08 | train_loss=0.0327 | val_loss=4.4355
Epoch 09 | train_loss=0.0418 | val_loss=4.9934
Epoch 10 | train_loss=0.0540 | val_loss=5.7416
Epoch 11 | train_loss=0.0113 | val_loss=6.1761
Epoch 12 | train_loss=0.0067 | val_loss=6.3000
Epoch 13 | train_loss=0.0236 | val_loss=6.9202
Epoch 14 | train_loss=0.0011 | val_loss=7.2531
Epoch 15 | train_loss=0.0011 | val_loss=7.5678
Epoch 16 | train_loss=0.0091 | val_loss=8.7823
Epoch 17 | train_loss=0.0277 | val_loss=9.5108
Epoch 18 | train_loss=0.0600 | val_loss=10.5605
Epoch 19 | train_loss=0.0300 | val_loss=11.3744
Epoch 20 | train_loss=0.0301 | val_loss=11.6486
Epoch 21 | train_loss=

### quick sanity check

In [30]:
# Test image model
model = AdImageModel(num_content_types=3, num_moods=4, freeze_encoder=False)
model.load_state_dict(torch.load(f'{DRIVE_PATH}/checkpoints/best_phase2.pt', map_location='cpu', weights_only=True))
model.eval()
print("Image model loaded OK")

# Test text encoder
encoder = TextEncoder()
vec = encoder.encode("Premium leather shoes crafted in Italy")
print(f"Text encoder OK — shape: {vec.shape}")

# Test caption generator
gen = CaptionGenerator()
caption = gen.generate("product showcase", "professional", "LinkedIn", "Premium leather shoes crafted in Italy")
print(f"Caption generator OK — {caption}")

print("\nAll src modules working")

Image model loaded OK


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Text encoder OK — shape: torch.Size([384])


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Caption generator OK — {'suggested_caption': '"Premium leather shoes crafted in Italy', 'keyword_suggestions': []}

All src modules working


In [31]:
# Copy MLflow DB if using local mode
db_source = f"{DRIVE_PATH}/mlflow.db"
db_backup = f"{DRIVE_PATH}/mlflow_backup.db"

if os.path.exists(db_source):
    shutil.copy2(db_source, db_backup)
    print("MLflow database backed up saved.")
else:
    print("MLflow DB not found.")

print("\nALL TRAINING COMPLETE!")

MLflow database backed up saved.

ALL TRAINING COMPLETE!
